# XL-Share: Comprehensive Evaluation for Manuscript

This notebook implements a rigorous evaluation of the **XL-Share** system, suitable for a research manuscript.

## Experimental Design
We perform a comprehensive evaluation covering:

### 1. Main Comparison & Ablation Study
We compare three system configurations to isolate the benefits of our contributions:
-   **Naive Offloading**: Synchronous copy from CPU to GPU on demand (Baseline).
-   **LRU-Only (No Prefetch)**: Caching with LRU eviction, but fetching is synchronous (Ablation).
-   **XL-Share (Full)**: Intelligent Asynchronous Prefetching + LRU Caching (Proposed).

### 2. Sensitivity Analysis
-   **Cache Size Sensitivity**: How performance changes as GPU memory becomes more constrained.
-   **Batch Size Sensitivity**: How computational intensity affects the system's ability to hide memory latency.
-   **Model Size Scaling**: Performance impact as the model size grows significantly larger than the cache.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import threading
import queue
import matplotlib.pyplot as plt
from collections import OrderedDict
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {DEVICE}")

## 1. Core System Components

In [ ]:
class CXLMemoryManager:
    def __init__(self):
        self.pool: Dict[str, torch.Tensor] = {}
        self.total_bytes = 0

    def store_weight(self, name: str, tensor: torch.Tensor):
        self.pool[name] = tensor.cpu().pin_memory()
        self.total_bytes += tensor.numel() * tensor.element_size()

    def get_weight(self, name: str) -> torch.Tensor:
        return self.pool[name]

class LocalCache:
    def __init__(self, capacity_mb: int = 1024):
        self.capacity_bytes = capacity_mb * 1024 * 1024
        self.current_bytes = 0
        self.cache: OrderedDict[str, torch.Tensor] = OrderedDict()
        self.lock = threading.RLock()
        self.hits = 0
        self.misses = 0

    def get(self, name: str) -> Optional[torch.Tensor]:
        with self.lock:
            if name in self.cache:
                tensor = self.cache.pop(name)
                self.cache[name] = tensor
                self.hits += 1
                return tensor
            self.misses += 1
            return None

    def put(self, name: str, tensor: torch.Tensor):
        with self.lock:
            size = tensor.numel() * tensor.element_size()
            if name in self.cache:
                old_tensor = self.cache.pop(name)
                self.current_bytes -= (old_tensor.numel() * old_tensor.element_size())
            
            while self.current_bytes + size > self.capacity_bytes and len(self.cache) > 0:
                self.cache.popitem(last=False)
                self.current_bytes -= (tensor.numel() * tensor.element_size()) 
            
            self.cache[name] = tensor
            self.current_bytes += size

    def reset_stats(self):
        self.hits = 0
        self.misses = 0

## 2. Configurable Engine (Supports Ablation)

In [ ]:
def functional_linear(input, weight, bias=None):
    return F.linear(input, weight, bias)

def functional_layer_norm(input, weight, bias, normalized_shape):
    return F.layer_norm(input, normalized_shape, weight, bias)

class XLShareEngine:
    def __init__(self, hidden_size, num_layers, vocab_size, cache_size_mb, 
                 enable_prefetch=True, enable_cache=True):
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.vocab_size = vocab_size
        self.enable_prefetch = enable_prefetch
        self.enable_cache = enable_cache
        
        self.mem_manager = CXLMemoryManager()
        # If cache disabled, set capacity to 0 (effectively)
        cap = cache_size_mb if enable_cache else 0
        self.local_cache = LocalCache(capacity_mb=cap)
        
        self._init_weights()
        self.layer_names = self._get_execution_order()
        
        self.stream = torch.cuda.Stream() if torch.cuda.is_available() else None
        self.active_transfers = {}
        self.curr_idx = 0

    def _init_weights(self):
        self.mem_manager.store_weight("emb.weight", torch.randn(self.vocab_size, self.hidden_size))
        for i in range(self.num_layers):
            self.mem_manager.store_weight(f"l{i}.ln1.weight", torch.ones(self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.ln1.bias", torch.zeros(self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.attn.q.weight", torch.randn(self.hidden_size, self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.attn.k.weight", torch.randn(self.hidden_size, self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.attn.v.weight", torch.randn(self.hidden_size, self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.attn.o.weight", torch.randn(self.hidden_size, self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.ln2.weight", torch.ones(self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.ln2.bias", torch.zeros(self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.ff1.weight", torch.randn(self.hidden_size * 4, self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.ff2.weight", torch.randn(self.hidden_size, self.hidden_size * 4))
        self.mem_manager.store_weight("head.weight", torch.randn(self.vocab_size, self.hidden_size))

    def _get_execution_order(self):
        order = ["emb.weight"]
        for i in range(self.num_layers):
            order.extend([
                f"l{i}.ln1.weight", f"l{i}.ln1.bias",
                f"l{i}.attn.q.weight", f"l{i}.attn.k.weight", f"l{i}.attn.v.weight", f"l{i}.attn.o.weight",
                f"l{i}.ln2.weight", f"l{i}.ln2.bias",
                f"l{i}.ff1.weight", f"l{i}.ff2.weight"
            ])
        order.append("head.weight")
        return order

    def prefetch(self, lookahead=5):
        if not self.enable_prefetch or not self.stream: return
        
        end_idx = min(len(self.layer_names), self.curr_idx + lookahead)
        with torch.cuda.stream(self.stream):
            for name in self.layer_names[self.curr_idx:end_idx]:
                if self.local_cache.get(name) is not None: continue
                
                cpu_tensor = self.mem_manager.get_weight(name)
                gpu_tensor = cpu_tensor.to(DEVICE, non_blocking=True)
                self.local_cache.put(name, gpu_tensor)
                event = torch.cuda.Event()
                event.record(self.stream)
                self.active_transfers[name] = event

    def get_weight(self, name: str) -> torch.Tensor:
        # Trigger prefetch
        self.prefetch()
        self.curr_idx += 1
        
        # Check cache
        tensor = self.local_cache.get(name)
        if tensor is not None:
            # Wait for async transfer if pending
            if name in self.active_transfers:
                self.active_transfers[name].wait()
                del self.active_transfers[name]
            return tensor
            
        # Cache Miss (Synchronous Fetch)
        cpu_tensor = self.mem_manager.get_weight(name)
        gpu_tensor = cpu_tensor.to(DEVICE)
        if self.enable_cache:
            self.local_cache.put(name, gpu_tensor)
        return gpu_tensor
    
    def forward(self, x_input):
        self.curr_idx = 0
        
        w_emb = self.get_weight("emb.weight")
        x = F.embedding(x_input, w_emb)
        
        for i in range(self.num_layers):
            w_ln1 = self.get_weight(f"l{i}.ln1.weight")
            b_ln1 = self.get_weight(f"l{i}.ln1.bias")
            residual = x
            x = functional_layer_norm(x, w_ln1, b_ln1, (self.hidden_size,))
            
            w_q = self.get_weight(f"l{i}.attn.q.weight")
            w_k = self.get_weight(f"l{i}.attn.k.weight")
            w_v = self.get_weight(f"l{i}.attn.v.weight")
            w_o = self.get_weight(f"l{i}.attn.o.weight")
            
            q = functional_linear(x, w_q)
            k = functional_linear(x, w_k)
            v = functional_linear(x, w_v)
            x = functional_linear(q, w_o) + residual
            
            w_ln2 = self.get_weight(f"l{i}.ln2.weight")
            b_ln2 = self.get_weight(f"l{i}.ln2.bias")
            residual = x
            x = functional_layer_norm(x, w_ln2, b_ln2, (self.hidden_size,))
            
            w_ff1 = self.get_weight(f"l{i}.ff1.weight")
            w_ff2 = self.get_weight(f"l{i}.ff2.weight")
            x = functional_linear(x, w_ff1)
            x = F.relu(x)
            x = functional_linear(x, w_ff2) + residual
            
        w_head = self.get_weight("head.weight")
        return functional_linear(x, w_head)

## 3. Experiment Runner

In [ ]:
def run_experiment(hidden_size=1024, cache_size_mb=256, batch_size=4, 
                   enable_prefetch=True, enable_cache=True, num_iters=5):
    
    engine = XLShareEngine(
        hidden_size=hidden_size, 
        num_layers=12, 
        vocab_size=50000, 
        cache_size_mb=cache_size_mb,
        enable_prefetch=enable_prefetch,
        enable_cache=enable_cache
    )
    
    x_input = torch.randint(0, 50000, (batch_size, 128)).to(DEVICE)
    
    # Warmup
    engine.forward(x_input)
    torch.cuda.synchronize()
    engine.local_cache.reset_stats()
    
    # Measure
    start = time.time()
    for _ in range(num_iters):
        engine.forward(x_input)
        torch.cuda.synchronize()
    total_time = time.time() - start
    
    avg_latency = (total_time / num_iters) * 1000
    throughput = (num_iters * batch_size * 128) / total_time
    hit_rate = engine.local_cache.hits / (engine.local_cache.hits + engine.local_cache.misses + 1e-6)
    
    return avg_latency, throughput, hit_rate

## 4. Scenario 1: Ablation Study (Cache Size Sensitivity)
We compare **Naive** (No Prefetch, No Cache reuse across layers effectively), **LRU-Only** (Cache but no prefetch), and **XL-Share**.

In [ ]:
cache_sizes = [128, 256, 512, 1024]
results_ablation = {
    'naive': [],
    'lru_only': [],
    'xlshare': []
}

print("Running Ablation Study...")
for size in cache_sizes:
    print(f"\nCache Size: {size}MB")
    
    # Naive: No Cache (Force eviction/small cache), No Prefetch
    # We simulate naive by disabling cache effectively or just using LRU without prefetch
    # Here we define Naive as: Cache Enabled (standard PyTorch behavior would keep in memory if possible, 
    # but here we mean 'Demand Paging'). So LRU-Only is essentially Demand Paging.
    # Let's define 'Naive' as strictly synchronous copy every time (Enable Cache=False).
    lat, _, _ = run_experiment(cache_size_mb=size, enable_prefetch=False, enable_cache=False)
    results_ablation['naive'].append(lat)
    print(f"  Naive (No Cache): {lat:.1f}ms")
    
    # LRU Only: Enable Cache, Disable Prefetch
    lat, _, _ = run_experiment(cache_size_mb=size, enable_prefetch=False, enable_cache=True)
    results_ablation['lru_only'].append(lat)
    print(f"  LRU Only:         {lat:.1f}ms")
    
    # XL-Share: Enable Cache, Enable Prefetch
    lat, _, _ = run_experiment(cache_size_mb=size, enable_prefetch=True, enable_cache=True)
    results_ablation['xlshare'].append(lat)
    print(f"  XL-Share:         {lat:.1f}ms")

## 5. Scenario 2: Batch Size Sensitivity
Does higher compute intensity (larger batch) help hide the latency better?

In [ ]:
batch_sizes = [1, 2, 4, 8, 16, 32]
results_batch = {'lru_only': [], 'xlshare': []}
FIXED_CACHE = 256 # Fixed cache size

print("\nRunning Batch Size Sensitivity...")
for b in batch_sizes:
    print(f"Batch Size: {b}")
    lat_lru, _, _ = run_experiment(cache_size_mb=FIXED_CACHE, batch_size=b, enable_prefetch=False)
    lat_xl, _, _ = run_experiment(cache_size_mb=FIXED_CACHE, batch_size=b, enable_prefetch=True)
    
    results_batch['lru_only'].append(lat_lru)
    results_batch['xlshare'].append(lat_xl)
    print(f"  LRU: {lat_lru:.1f}ms | XL: {lat_xl:.1f}ms | Speedup: {lat_lru/lat_xl:.2f}x")

## 6. Scenario 3: Model Size Scaling
How does the system perform as we increase the hidden dimension (model size)?

In [ ]:
hidden_sizes = [768, 1024, 1536, 2048]
results_model = {'lru_only': [], 'xlshare': []}

print("\nRunning Model Size Scaling...")
for h in hidden_sizes:
    print(f"Hidden Size: {h}")
    lat_lru, _, _ = run_experiment(hidden_size=h, cache_size_mb=FIXED_CACHE, enable_prefetch=False)
    lat_xl, _, _ = run_experiment(hidden_size=h, cache_size_mb=FIXED_CACHE, enable_prefetch=True)
    
    results_model['lru_only'].append(lat_lru)
    results_model['xlshare'].append(lat_xl)
    print(f"  LRU: {lat_lru:.1f}ms | XL: {lat_xl:.1f}ms")

## 7. Visualization

In [ ]:
plt.figure(figsize=(18, 5))

# 1. Ablation
plt.subplot(1, 3, 1)
plt.plot(cache_sizes, results_ablation['naive'], 'o--', label='Naive (No Cache)', color='red')
plt.plot(cache_sizes, results_ablation['lru_only'], 's--', label='LRU Only', color='gray')
plt.plot(cache_sizes, results_ablation['xlshare'], '^-', label='XL-Share', color='blue', linewidth=2)
plt.xlabel('Cache Size (MB)')
plt.ylabel('Latency (ms)')
plt.title('Ablation Study')
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Batch Sensitivity
plt.subplot(1, 3, 2)
plt.plot(batch_sizes, results_batch['lru_only'], 's--', label='LRU Only', color='gray')
plt.plot(batch_sizes, results_batch['xlshare'], '^-', label='XL-Share', color='blue', linewidth=2)
plt.xlabel('Batch Size')
plt.ylabel('Latency (ms)')
plt.title('Batch Size Sensitivity')
plt.legend()
plt.grid(True, alpha=0.3)

# 3. Model Scaling
plt.subplot(1, 3, 3)
plt.plot(hidden_sizes, results_model['lru_only'], 's--', label='LRU Only', color='gray')
plt.plot(hidden_sizes, results_model['xlshare'], '^-', label='XL-Share', color='blue', linewidth=2)
plt.xlabel('Hidden Size')
plt.ylabel('Latency (ms)')
plt.title('Model Size Scaling')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()